In [1]:
# hybrid_face_recognition.py
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib
from scipy.spatial.distance import cosine

# ----------------------------
# 1️⃣ Setup DL face detector + embedding model
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ----------------------------
# 2️⃣ Simple Python neuron network
# ----------------------------
class PyNeuron:
    def __init__(self, embedding, person_name):
        self.embedding = embedding
        self.person_name = person_name
        self.links = set()

class PyNeuronNetwork:
    def __init__(self):
        self.neurons = []

    def add_neuron(self, embedding, person_name):
        n = PyNeuron(embedding, person_name)
        # link neuron to similar existing neurons
        for other in self.neurons:
            sim = 1 - cosine(n.embedding, other.embedding)
            if sim > 0.85:  # threshold for linking
                n.links.add(other)
                other.links.add(n)
        self.neurons.append(n)

    def search_similar(self, embedding, top_k=1):
        best = None
        max_sim = 0
        for neuron in self.neurons:
            sim = 1 - cosine(embedding, neuron.embedding)
            if sim > max_sim:
                max_sim = sim
                best = neuron
        return best, max_sim

neuron_net = PyNeuronNetwork()

# ----------------------------
# 3️⃣ Load dataset (train/test)
# ----------------------------
def load_data(folder_path):
    X = []
    y = []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    emb = get_face_embedding(img_path)
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
                        neuron_net.add_neuron(emb, person_name)  # add to neuron network
    return np.array(X), np.array(y)

train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

X_train, y_train = load_data(train_folder)
X_test, y_test = load_data(test_folder)
print(f"✅ Loaded {len(X_train)} train samples and {len(X_test)} test samples")

# ----------------------------
# 4️⃣ Scale embeddings for SVM
# ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ----------------------------
# 5️⃣ Train SVM classifier
# ----------------------------
svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train_scaled, y_train)

# ----------------------------
# 6️⃣ Evaluate baseline SVM
# ----------------------------
y_pred = svm_clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ SVM Test Accuracy: {acc*100:.2f}%")

# ----------------------------
# 7️⃣ Hybrid prediction function
# ----------------------------
def hybrid_predict(img_path, similarity_threshold=0.85):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    # 1. SVM prediction
    svm_label = svm_clf.predict(scaler.transform([emb]))[0]

    # 2. Neuron similarity search
    neuron, sim = neuron_net.search_similar(emb)
    if neuron and sim >= similarity_threshold:
        return neuron.person_name

    return svm_label

# ----------------------------
# 8️⃣ Example usage
# ----------------------------
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img1.jpg"
predicted_person = hybrid_predict(image_path)
print(f"Predicted person (hybrid): {predicted_person}")

# ----------------------------
# 9️⃣ Save models
# ----------------------------
joblib.dump(svm_clf, "svm_face_recognition.pkl")
joblib.dump(scaler, "svm_scaler.pkl")
print("✅ SVM model and scaler saved!")

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train samples and 372 test samples
✅ SVM Test Accuracy: 99.73%
Predicted person (hybrid): Akshay Kumar
✅ SVM model and scaler saved!


In [3]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
import sys

print(sys.version)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch Version: 2.5.1
CUDA Available: True
CUDA Version: 12.1
3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [7]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ---------------------------- Neuron Brain ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)  # simple unique ID
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = set()

    def update_confidence(self, delta):
        self.confidence += delta
        if self.confidence < 0: self.confidence = 0

    def increment_mistakes(self):
        self.mistakes += 1

class NeuronNetwork:
    def __init__(self):
        self.neurons = {}

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
    
        # Add to network first
        self.neurons[n.id] = n
    
        # Then link to similar neurons
        for nid, neuron in self.neurons.items():
            if nid == n.id:
                continue  # skip linking to itself
            if np.linalg.norm(neuron.embedding - embedding) < 0.6:
                self.link_neurons(n.id, nid)
    
        return n.id

    def find_similar(self, embedding, threshold=0.6):
        for nid, neuron in self.neurons.items():
            if np.linalg.norm(neuron.embedding - embedding) < threshold:
                return nid
        return None

    def update_neuron(self, nid, correct=True):
        if correct:
            self.neurons[nid].update_confidence(+1)
        else:
            self.neurons[nid].update_confidence(-1)
            self.neurons[nid].increment_mistakes()

    def link_neurons(self, nid1, nid2):
        self.neurons[nid1].links.add(nid2)
        self.neurons[nid2].links.add(nid1)

    def print_summary(self):
        for nid, n in self.neurons.items():
            print(f"Neuron {nid}: label={n.label}, conf={n.confidence}, mistakes={n.mistakes}, links={n.links}")

# ---------------------------- Load Dataset ----------------------------
def load_data(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    emb = get_face_embedding(img_path)
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)

train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

X_train, y_train = load_data(train_folder)
X_test, y_test = load_data(test_folder)

print(f"✅ Loaded {len(X_train)} train samples and {len(X_test)} test samples")

# ---------------------------- Scale Embeddings & Train SVM ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train_scaled, y_train)

y_pred = svm_clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ SVM Test Accuracy: {acc*100:.2f}%")

joblib.dump(svm_clf, "svm_face_recognition.pkl")
joblib.dump(scaler, "svm_scaler.pkl")
print("✅ SVM model and scaler saved!")

# ---------------------------- Initialize Neuron Brain ----------------------------
neuron_net = NeuronNetwork()
for emb, label in zip(X_train, y_train):
    neuron_net.create_neuron(emb, label)

# ---------------------------- Prediction + Reinforcement ----------------------------
def predict_and_reinforce(img_path, true_label=None):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    # SVM prediction
    pred_label = svm_clf.predict(scaler.transform([emb]))[0]

    # Neuron brain update
    nid = neuron_net.find_similar(emb)
    if nid:
        correct = (pred_label == true_label) if true_label else True
        neuron_net.update_neuron(nid, correct)
    else:
        # New neuron if unseen
        neuron_net.create_neuron(emb, true_label if true_label else pred_label)

    return pred_label

# ---------------------------- Example Usage ----------------------------
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img1.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Akshay Kumar")
print(f"Predicted person (hybrid): {predicted_person}")

# Print neuron brain summary
neuron_net.print_summary()

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train samples and 372 test samples
✅ SVM Test Accuracy: 99.73%
✅ SVM model and scaler saved!
Predicted person (hybrid): Akshay Kumar
Neuron 200033336: label=Akshay Kumar, conf=1.0, mistakes=0, links=set()
Neuron 799388065: label=Akshay Kumar, conf=1.0, mistakes=0, links={35076296, 922345283}
Neuron 972863700: label=Akshay Kumar, conf=1.0, mistakes=0, links={429435985, 282192433, 367535894}
Neuron 832726185: label=Akshay Kumar, conf=1.0, mistakes=0, links={376778497, 922345283, 672815563, 249790231, 131830559}
Neuron 533085741: label=Akshay Kumar, conf=1.0, mistakes=0, links={24082186, 74827606, 967217663}
Neuron 987082338: label=Akshay Kumar, conf=1.0, mistakes=0, links=set()
Neuron 964979373: label=Akshay Kumar, conf=1.0, mistakes=0, links={376778497, 880359803, 131830559}
Neuron 899591654: label=Akshay Kumar, conf=1.0, mistakes=0, links={388520406, 131830559}
Neuron 174171628: label=Akshay Kumar, conf=2.0, mistakes=0, links=set()
Neuron 203780248: label=Akshay Kumar, co

In [12]:
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img3.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Elizabeth Olsen")
print(f"Predicted person (hybrid): {predicted_person}")


Predicted person (hybrid): Robert Downey Jr


In [13]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ---------------------------- Advanced Neuron Brain ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = set()
        self.samples = [embedding]

    def update_confidence(self, delta):
        self.confidence = max(0.0, self.confidence + delta)

    def increment_mistakes(self):
        self.mistakes += 1

    def merge_embedding(self, new_emb):
        self.samples.append(new_emb)
        self.embedding = np.mean(self.samples, axis=0)

    def decay_confidence(self, factor=0.99):
        self.confidence *= factor

class NeuronNetwork:
    def __init__(self, base_threshold=0.6):
        self.neurons = {}
        self.base_threshold = base_threshold

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
        self.neurons[n.id] = n

        # Link and merge similar neurons
        for nid, neuron in list(self.neurons.items()):
            if nid == n.id:
                continue
            dist = np.linalg.norm(neuron.embedding - embedding)
            dynamic_thresh = self.base_threshold * (2.0 - neuron.confidence)
            if dist < dynamic_thresh and neuron.label == label:
                self.link_neurons(n.id, nid)
                neuron.merge_embedding(embedding)
                n.merge_embedding(neuron.embedding)

        return n.id

    def find_similar(self, embedding, threshold=None):
        if threshold is None:
            threshold = self.base_threshold
        best_nid = None
        min_dist = float('inf')
        for nid, neuron in self.neurons.items():
            dist = np.linalg.norm(neuron.embedding - embedding)
            if dist < threshold and dist < min_dist:
                min_dist = dist
                best_nid = nid
        return best_nid

    def update_neuron(self, nid, correct=True):
        if correct:
            self.neurons[nid].update_confidence(+1)
        else:
            self.neurons[nid].update_confidence(-1)
            self.neurons[nid].increment_mistakes()

    def link_neurons(self, nid1, nid2):
        self.neurons[nid1].links.add(nid2)
        self.neurons[nid2].links.add(nid1)

    def hybrid_predict(self, embedding, svm_model, scaler):
        """Combine SVM and neuron confidence for robust prediction"""
        svm_pred = svm_model.predict(scaler.transform([embedding]))[0]
        nid = self.find_similar(embedding)
        if nid:
            neuron = self.neurons[nid]
            if neuron.confidence >= 1.5:  # neuron confidence > threshold → trust neuron
                return neuron.label
        return svm_pred

    def decay_all_confidence(self, factor=0.995):
        for n in self.neurons.values():
            n.decay_confidence(factor)

    def print_summary(self):
        for nid, n in self.neurons.items():
            print(f"Neuron {nid}: label={n.label}, conf={n.confidence:.2f}, mistakes={n.mistakes}, links={n.links}, samples={len(n.samples)}")

# ---------------------------- Load Dataset ----------------------------
def load_data(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    emb = get_face_embedding(img_path)
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)

train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

X_train, y_train = load_data(train_folder)
X_test, y_test = load_data(test_folder)
print(f"✅ Loaded {len(X_train)} train samples and {len(X_test)} test samples")

# ---------------------------- Scale & Train SVM ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train_scaled, y_train)

y_pred = svm_clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ SVM Test Accuracy: {acc*100:.2f}%")

joblib.dump(svm_clf, "svm_face_recognition.pkl")
joblib.dump(scaler, "svm_scaler.pkl")
print("✅ SVM model and scaler saved!")

# ---------------------------- Initialize Neuron Brain ----------------------------
neuron_net = NeuronNetwork()
for emb, label in zip(X_train, y_train):
    neuron_net.create_neuron(emb, label)

# ---------------------------- Prediction + Auto-update ----------------------------
def predict_and_reinforce(img_path, true_label=None, auto_update=True):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    pred_label = neuron_net.hybrid_predict(emb, svm_clf, scaler)

    # Auto-update / reinforcement
    if auto_update:
        nid = neuron_net.find_similar(emb)
        if nid:
            correct = (pred_label == true_label) if true_label else True
            neuron_net.update_neuron(nid, correct)
        else:
            neuron_net.create_neuron(emb, true_label if true_label else pred_label)
        neuron_net.decay_all_confidence()  # decay old neurons slowly

    return pred_label

# ---------------------------- Example Usage ----------------------------
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img1.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Akshay Kumar")
print(f"Predicted person (hybrid): {predicted_person}")

# Print neuron brain summary
neuron_net.print_summary()

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train samples and 372 test samples
✅ SVM Test Accuracy: 99.73%
✅ SVM model and scaler saved!
Predicted person (hybrid): Akshay Kumar
Neuron 202599714: label=Akshay Kumar, conf=0.99, mistakes=0, links=set(), samples=1
Neuron 306418321: label=Akshay Kumar, conf=0.99, mistakes=0, links={194749411, 603287141, 811143051, 105358221, 43245455, 539268303, 348425144, 579045533}, samples=9
Neuron 639352494: label=Akshay Kumar, conf=0.99, mistakes=0, links={909010764, 716196133, 622450102}, samples=4
Neuron 304750190: label=Akshay Kumar, conf=0.99, mistakes=0, links={858693250, 194749411, 649117221, 603287141, 811143051, 711540109, 105358221, 43245455, 87776560, 539268303, 615114742, 348425144, 63695449, 579045533}, samples=15
Neuron 206418996: label=Akshay Kumar, conf=0.99, mistakes=0, links={858693250, 194749411, 649117221, 603287141, 95654123, 811143051, 105358221, 43245455, 87776560, 539268303, 615114742, 348425144, 70985208, 63695449, 579045533}, samples=16
Neuron 453230321: la

In [14]:
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img3.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Akshay Kumar")
print(f"Predicted person (hybrid): {predicted_person}")

Predicted person (hybrid): Robert Downey Jr


In [15]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib
import time

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ---------------------------- Advanced Neuron Brain ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = {}  # neuron_id: weight
        self.last_seen = time.time()

    def update_confidence(self, delta):
        self.confidence += delta
        if self.confidence < 0:
            self.confidence = 0
        self.last_seen = time.time()

    def increment_mistakes(self):
        self.mistakes += 1
        self.last_seen = time.time()

    def decay(self, decay_rate=0.001):
        # decay confidence over time
        elapsed = time.time() - self.last_seen
        self.confidence -= decay_rate * elapsed
        if self.confidence < 0:
            self.confidence = 0

class NeuronNetwork:
    def __init__(self):
        self.neurons = {}

    def cosine_similarity(self, emb1, emb2):
        return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-10)

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
        self.neurons[n.id] = n

        # link to similar neurons
        for nid, neuron in self.neurons.items():
            if nid == n.id:
                continue
            sim = self.cosine_similarity(n.embedding, neuron.embedding)
            if sim > 0.7:
                self.link_neurons(n.id, nid, weight=sim)
        return n.id

    def find_similar(self, embedding, threshold=0.7):
        best_nid = None
        best_sim = -1
        for nid, neuron in self.neurons.items():
            sim = self.cosine_similarity(neuron.embedding, embedding)
            if sim > threshold and sim > best_sim:
                best_nid = nid
                best_sim = sim
        return best_nid

    def update_neuron(self, nid, correct=True, new_embedding=None):
        neuron = self.neurons[nid]
        if correct:
            neuron.update_confidence(+1)
        else:
            neuron.update_confidence(-1)
            neuron.increment_mistakes()
        if new_embedding is not None:
            # slowly adjust embedding toward new input
            neuron.embedding = 0.9 * neuron.embedding + 0.1 * new_embedding

    def link_neurons(self, nid1, nid2, weight=1.0):
        self.neurons[nid1].links[nid2] = weight
        self.neurons[nid2].links[nid1] = weight

    def decay_and_prune(self, min_confidence=0.1):
        to_delete = []
        for nid, neuron in self.neurons.items():
            neuron.decay()
            if neuron.confidence < min_confidence:
                to_delete.append(nid)
        for nid in to_delete:
            # remove links
            for linked_id in self.neurons[nid].links.keys():
                if linked_id in self.neurons:
                    self.neurons[linked_id].links.pop(nid, None)
            self.neurons.pop(nid)

    def print_summary(self):
        for nid, n in self.neurons.items():
            print(f"Neuron {nid}: label={n.label}, conf={n.confidence:.2f}, mistakes={n.mistakes}, links={len(n.links)}")

# ---------------------------- Load Dataset ----------------------------
def load_data(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    emb = get_face_embedding(img_path)
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)

train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

X_train, y_train = load_data(train_folder)
X_test, y_test = load_data(test_folder)

print(f"✅ Loaded {len(X_train)} train samples and {len(X_test)} test samples")

# ---------------------------- Scale Embeddings & Train SVM ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train_scaled, y_train)

y_pred = svm_clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ SVM Test Accuracy: {acc*100:.2f}%")

joblib.dump(svm_clf, "svm_face_recognition.pkl")
joblib.dump(scaler, "svm_scaler.pkl")
print("✅ SVM model and scaler saved!")

# ---------------------------- Initialize Neuron Brain ----------------------------
neuron_net = NeuronNetwork()
for emb, label in zip(X_train, y_train):
    neuron_net.create_neuron(emb, label)

# ---------------------------- Prediction + Reinforcement ----------------------------
def predict_and_reinforce(img_path, true_label=None, online_svm_update=False):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    # SVM prediction
    pred_label = svm_clf.predict(scaler.transform([emb]))[0]

    # Neuron brain update
    nid = neuron_net.find_similar(emb)
    if nid:
        correct = (pred_label == true_label) if true_label else True
        neuron_net.update_neuron(nid, correct, new_embedding=emb)
    else:
        neuron_net.create_neuron(emb, true_label if true_label else pred_label)

    # Optional: online SVM update
    if online_svm_update and true_label:
        X_online = scaler.transform([emb])
        y_online = [true_label]
        svm_clf.fit(np.vstack([X_train_scaled, X_online]), np.hstack([y_train, y_online]))

    # Decay & prune neurons periodically
    neuron_net.decay_and_prune()

    return pred_label

# ---------------------------- Example Usage ----------------------------
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img1.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Akshay Kumar", online_svm_update=True)
print(f"Predicted person (hybrid): {predicted_person}")

# Print neuron brain summary
neuron_net.print_summary()

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train samples and 372 test samples
✅ SVM Test Accuracy: 99.73%
✅ SVM model and scaler saved!
Predicted person (hybrid): Akshay Kumar
Neuron 626404092: label=Akshay Kumar, conf=0.98, mistakes=0, links=0
Neuron 514542609: label=Akshay Kumar, conf=0.98, mistakes=0, links=19
Neuron 273681444: label=Akshay Kumar, conf=0.98, mistakes=0, links=6
Neuron 844189932: label=Akshay Kumar, conf=0.98, mistakes=0, links=26
Neuron 843886430: label=Akshay Kumar, conf=0.98, mistakes=0, links=22
Neuron 242485262: label=Akshay Kumar, conf=0.98, mistakes=0, links=16
Neuron 774494310: label=Akshay Kumar, conf=0.98, mistakes=0, links=15
Neuron 173624142: label=Akshay Kumar, conf=0.98, mistakes=0, links=25
Neuron 936762003: label=Akshay Kumar, conf=2.00, mistakes=0, links=0
Neuron 452956944: label=Akshay Kumar, conf=0.98, mistakes=0, links=21
Neuron 151205927: label=Akshay Kumar, conf=0.98, mistakes=0, links=18
Neuron 904870260: label=Akshay Kumar, conf=0.98, mistakes=0, links=27
Neuron 333645508

In [17]:
image_path = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img3.jpg"
predicted_person = predict_and_reinforce(image_path, true_label="Akshay Kumar", online_svm_update=True)
print(f"Predicted person (hybrid): {predicted_person}")

Predicted person (hybrid): Robert Downey Jr


In [18]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib
import pickle

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ---------------------------- Neuron Brain ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = {}  # neuron_id -> weight

    def update_confidence(self, delta):
        self.confidence += delta
        if self.confidence < 0:
            self.confidence = 0

    def increment_mistakes(self):
        self.mistakes += 1

    def decay(self, decay_rate=0.01):
        self.confidence -= decay_rate
        if self.confidence < 0:
            self.confidence = 0
        # decay link weights
        for nid in list(self.links.keys()):
            self.links[nid] *= (1 - decay_rate)
            if self.links[nid] < 0.05:
                self.links.pop(nid)

class NeuronNetwork:
    def __init__(self):
        self.neurons = {}

    def cosine_similarity(self, emb1, emb2):
        return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-10)

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
        self.neurons[n.id] = n
        # Link to similar neurons
        for nid, neuron in self.neurons.items():
            if nid == n.id:
                continue
            sim = self.cosine_similarity(n.embedding, neuron.embedding)
            if sim > 0.7:
                self.link_neurons(n.id, nid, weight=sim)
        return n.id

    def find_similar(self, embedding, threshold=0.7):
        best_nid = None
        best_sim = -1
        for nid, neuron in self.neurons.items():
            sim = self.cosine_similarity(neuron.embedding, embedding)
            if sim > threshold and sim > best_sim:
                best_nid = nid
                best_sim = sim
        return best_nid

    def update_neuron(self, nid, correct=True, new_embedding=None):
        neuron = self.neurons[nid]
        if correct:
            neuron.update_confidence(+1)
        else:
            neuron.update_confidence(-1)
            neuron.increment_mistakes()
        if new_embedding is not None:
            neuron.embedding = 0.9 * neuron.embedding + 0.1 * new_embedding
        # reinforce links
        for linked_id in neuron.links.keys():
            self.neurons[linked_id].update_confidence(+0.5)
            neuron.links[linked_id] = min(neuron.links[linked_id] + 0.05, 1.0)

    def link_neurons(self, nid1, nid2, weight=1.0):
        self.neurons[nid1].links[nid2] = weight
        self.neurons[nid2].links[nid1] = weight

    def decay_and_prune(self, min_confidence=0.1):
        to_delete = []
        for nid, neuron in self.neurons.items():
            neuron.decay()
            if neuron.confidence < min_confidence:
                to_delete.append(nid)
        for nid in to_delete:
            for linked_id in list(self.neurons[nid].links.keys()):
                if linked_id in self.neurons:
                    self.neurons[linked_id].links.pop(nid, None)
            self.neurons.pop(nid)

    def predict(self, embedding, consider_links=True):
        nid = self.find_similar(embedding)
        if nid is None:
            return None
        neuron = self.neurons[nid]
        if consider_links and neuron.links:
            label_scores = {neuron.label: neuron.confidence}
            for linked_id, weight in neuron.links.items():
                linked_neuron = self.neurons[linked_id]
                label_scores[linked_neuron.label] = label_scores.get(linked_neuron.label, 0) + linked_neuron.confidence * weight
            return max(label_scores, key=label_scores.get)
        return neuron.label

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

# ---------------------------- Load Dataset ----------------------------
def load_data(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    emb = get_face_embedding(img_path)
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)

# ---------------------------- Paths ----------------------------
train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

# ---------------------------- Load & Train ----------------------------
X_train, y_train = load_data(train_folder)
X_test, y_test = load_data(test_folder)

print(f"✅ Loaded {len(X_train)} train samples and {len(X_test)} test samples")

# Scale embeddings
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM
svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train_scaled, y_train)
y_pred = svm_clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ SVM Test Accuracy: {acc*100:.2f}%")

# Save SVM
joblib.dump(svm_clf, "svm_face_recognition.pkl")
joblib.dump(scaler, "svm_scaler.pkl")
print("✅ SVM model and scaler saved!")

# ---------------------------- Initialize Neuron Network ----------------------------
neuron_net = NeuronNetwork()
for emb, label in zip(X_train, y_train):
    neuron_net.create_neuron(emb, label)

# ---------------------------- Prediction + Auto-Reinforcement ----------------------------
def predict_and_reinforce(img_path, true_label=None, auto_update=True):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    # 1. SVM prediction
    pred_label = svm_clf.predict(scaler.transform([emb]))[0]

    # 2. NeuronNetwork prediction
    neuron_pred = neuron_net.predict(emb)
    final_pred = neuron_pred if neuron_pred else pred_label

    # 3. Auto-update NeuronNetwork
    if auto_update:
        nid = neuron_net.find_similar(emb)
        if nid:
            correct = (final_pred == true_label) if true_label else True
            neuron_net.update_neuron(nid, correct, new_embedding=emb)
        else:
            neuron_net.create_neuron(emb, true_label if true_label else final_pred)

    # 4. Decay & prune occasionally
    neuron_net.decay_and_prune()

    return final_pred

# ---------------------------- Example Usage ----------------------------
example_image = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img1.jpg"
predicted_person = predict_and_reinforce(example_image, true_label="Akshay Kumar")
print(f"Predicted person (hybrid RL-style): {predicted_person}")

# Print neuron summary
neuron_net.decay_and_prune()
for nid, n in neuron_net.neurons.items():
    print(f"Neuron {nid}: label={n.label}, conf={n.confidence}, mistakes={n.mistakes}, links={n.links}")

# ---------------------------- Save Neuron Network ----------------------------
neuron_net.save("neuron_network.pkl")
print("✅ NeuronNetwork saved!")

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train samples and 372 test samples
✅ SVM Test Accuracy: 99.73%
✅ SVM model and scaler saved!


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



✅ NeuronNetwork saved!


In [20]:
example_image = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test\img4.jpg"
predicted_person = predict_and_reinforce(example_image, true_label="Akshay Kumar")
print(f"Predicted person (hybrid RL-style): {predicted_person}")


Predicted person (hybrid RL-style): Elizabeth Olsen


In [2]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib
import sqlite3
import json

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()

# ---------------------------- Neuron for Face ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = {}

    def update_confidence(self, delta):
        self.confidence = max(0.0, self.confidence + delta)

    def increment_mistakes(self):
        self.mistakes += 1

    def decay(self, rate=0.99):
        self.confidence *= rate

# ---------------------------- Neuron Network ----------------------------
class NeuronNetwork:
    def __init__(self, db_path=None):
        self.neurons = {}
        self.db_path = db_path
        if db_path:
            self._init_db()

    @staticmethod
    def cosine_similarity(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
        self.neurons[n.id] = n
        # Link similar neurons
        for nid, neuron in self.neurons.items():
            if nid == n.id: 
                continue
            sim = self.cosine_similarity(n.embedding, neuron.embedding)
            if sim > 0.7:
                self.link_neurons(n.id, nid, weight=sim)
        return n.id

    def find_similar(self, embedding, threshold=0.7):
        best_nid = None
        best_sim = -1
        for nid, neuron in self.neurons.items():
            sim = self.cosine_similarity(neuron.embedding, embedding)
            if sim > threshold and sim > best_sim:
                best_sim = sim
                best_nid = nid
        return best_nid

    def update_neuron(self, nid, correct=True, new_embedding=None):
        neuron = self.neurons[nid]
        if correct:
            neuron.update_confidence(+1)
        else:
            neuron.update_confidence(-1)
            neuron.increment_mistakes()
        if new_embedding is not None:
            neuron.embedding = 0.9 * neuron.embedding + 0.1 * new_embedding

    def link_neurons(self, nid1, nid2, weight=1.0):
        self.neurons[nid1].links[nid2] = weight
        self.neurons[nid2].links[nid1] = weight

    # Decay and prune
    def decay_and_prune(self, min_confidence=0.1):
        to_delete = []
        for nid, neuron in self.neurons.items():
            neuron.decay()
            if neuron.confidence < min_confidence:
                to_delete.append(nid)
        for nid in to_delete:
            for linked_id in list(self.neurons[nid].links.keys()):
                if linked_id in self.neurons:
                    self.neurons[linked_id].links.pop(nid, None)
            self.neurons.pop(nid)

    # ---------------------------- SQLite persistence ----------------------------
    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, timeout=10)
        self.conn.execute("PRAGMA journal_mode=WAL;")  # better concurrency
        c = self.conn.cursor()
        c.execute('''
            CREATE TABLE IF NOT EXISTS neurons (
                id INTEGER PRIMARY KEY,
                label TEXT,
                confidence REAL,
                mistakes INTEGER,
                embedding TEXT,
                links TEXT
            )
        ''')
        self.conn.commit()

    import time

    def save(self, retries=5):
        if not self.db_path:
            raise ValueError("No db_path set for saving.")

        for attempt in range(retries):
            try:
                c = self.conn.cursor()
                for nid, neuron in self.neurons.items():
                    emb_json = json.dumps(neuron.embedding.tolist())
                    links_dict = {str(k): float(v) for k, v in neuron.links.items()}
                    links_json = json.dumps(links_dict)

                    c.execute('''
                        INSERT OR REPLACE INTO neurons (id, label, confidence, mistakes, embedding, links)
                        VALUES (?, ?, ?, ?, ?, ?)
                    ''', (nid, neuron.label, neuron.confidence, neuron.mistakes, emb_json, links_json))

                self.conn.commit()
                return  # success

            except sqlite3.OperationalError as e:
                if "locked" in str(e):
                    print(f"⚠️ DB locked, retrying... ({attempt+1})")
                    time.sleep(1)
                else:
                    raise e
    def close(self):
        if hasattr(self, "conn"):
            self.conn.close()

    def load(self):
        if not self.db_path:
            raise ValueError("No db_path set for loading.")
        c = self.conn.cursor()
        c.execute('SELECT id, label, confidence, mistakes, embedding, links FROM neurons')
        rows = c.fetchall()
        self.neurons = {}
        for row in rows:
            nid, label, conf, mistakes, emb_json, links_json = row
            neuron = Neuron(np.array(json.loads(emb_json)), label)
            neuron.id = nid
            neuron.confidence = conf
            neuron.mistakes = mistakes
            neuron.links = json.loads(links_json)
            self.neurons[nid] = neuron

    def print_summary(self):
        for nid, n in self.neurons.items():
            print(f"Neuron {nid}: label={n.label}, conf={n.confidence:.2f}, mistakes={n.mistakes}, links={list(n.links.keys())}")

# ---------------------------- Face Recognition + RL ----------------------------
def load_dataset(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    emb = get_face_embedding(os.path.join(person_path, img_file))
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)

def train_svm(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    clf = SVC(kernel='rbf', probability=True)
    clf.fit(X_scaled, y_train)
    return clf, scaler

def predict_face(nn: NeuronNetwork, svm_clf, scaler, img_path, true_label=None):
    emb = get_face_embedding(img_path)
    if emb is None:
        return "No face detected"

    # SVM prediction
    pred_label = svm_clf.predict(scaler.transform([emb]))[0]

    # RL update in neuron brain
    nid = nn.find_similar(emb)
    if nid:
        correct = (pred_label == true_label) if true_label else True
        nn.update_neuron(nid, correct=correct, new_embedding=emb)
    else:
        nn.create_neuron(emb, true_label if true_label else pred_label)

    return pred_label

# ---------------------------- Example Usage ----------------------------
if __name__ == "__main__":
    train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
    test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

    # Load dataset
    X_train, y_train = load_dataset(train_folder)
    X_test, y_test = load_dataset(test_folder)
    print(f"✅ Loaded {len(X_train)} train and {len(X_test)} test images")

    # Train SVM
    svm_clf, scaler = train_svm(X_train, y_train)
    y_pred = svm_clf.predict(scaler.transform(X_test))
    print(f"✅ SVM Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

    # Save SVM model & scaler
    joblib.dump(svm_clf, "svm_face_recognition.pkl")
    joblib.dump(scaler, "svm_scaler.pkl")
    print("✅ SVM model and scaler saved!")

    # Initialize Neuron Brain
    nn = NeuronNetwork(db_path="face_neurons.sqlite")
    for emb, label in zip(X_train, y_train):
        nn.create_neuron(emb, label)

    # Test prediction + RL
    example_img = os.path.join(test_folder, "img1.jpg")
    pred = predict_face(nn, svm_clf, scaler, example_img, true_label="Akshay Kumar")
    print(f"Predicted person (hybrid RL): {pred}")

    # Save neuron brain
    nn.save()
    nn.close()
    nn.print_summary()

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_p

✅ Loaded 2189 train and 372 test images
✅ SVM Test Accuracy: 99.73%
✅ SVM model and scaler saved!
Predicted person (hybrid RL): Akshay Kumar
Neuron 287077573: label=Akshay Kumar, conf=1.00, mistakes=0, links=[]
Neuron 58364863: label=Akshay Kumar, conf=1.00, mistakes=0, links=[44022772, 623746047, 795836291, 781020506, 648665981, 261228973, 260083081, 447279208, 844619399, 694984376, 604908360, 566988297, 412132981, 457915340, 961677176, 65289977, 444234592, 96158855, 124187954]
Neuron 453018674: label=Akshay Kumar, conf=1.00, mistakes=0, links=[795836291, 562368468, 457915340, 937744903, 831975466, 794790104]
Neuron 411344678: label=Akshay Kumar, conf=1.00, mistakes=0, links=[995695543, 44022772, 623746047, 795836291, 781020506, 658342517, 648665981, 261228973, 231110787, 260083081, 447279208, 844619399, 501595274, 694984376, 604908360, 566988297, 412132981, 406005318, 696029450, 811410212, 961677176, 65289977, 444234592, 96158855, 310564195, 124187954]
Neuron 995695543: label=Akshay 

In [4]:
    example_img = os.path.join(test_folder, "images.jpg")
    pred = predict_face(nn, svm_clf, scaler, example_img, true_label="")
    print(f"Predicted person (hybrid RL): {pred}")

Predicted person (hybrid RL): Akshay Kumar


In [22]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import joblib
import sqlite3
import json

# ---------------------------- DL Face Models ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=10, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_face_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face)
    return embedding.squeeze(0).cpu().numpy()
class SearchEngine:
    def search(self, query):
        # placeholder results
        return [
            {
                "url": "https://example.com/person1.jpg",
                "image_embedding": np.random.rand(512),
            },
            {
                "url": "https://example.com/person2.jpg",
                "image_embedding": np.random.rand(512),
            },
            {
                "url": "https://example.com/person3.jpg",
                "image_embedding": np.random.rand(512),
            },
        ]  
class CandidateIdentity:
    def __init__(self, name="UNKNOWN"):
        self.id = np.random.randint(1e6, 1e9)
        self.name = name
        self.confidence = 0.0
        self.sources = []
        self.embeddings = []
        self.status = "PENDING"

    def add_source(self, url):
        if url not in self.sources:
            self.sources.append(url)

    def add_embedding(self, embedding):
        self.embeddings.append(embedding)

    def update_confidence(self, delta):
        self.confidence = max(0.0, min(1.0, self.confidence + delta))

    def promote(self):
        self.status = "VERIFIED"
# ---------------------------- Neuron for Face ----------------------------
class Neuron:
    def __init__(self, embedding, label):
        self.id = np.random.randint(1e6, 1e9)
        self.embedding = embedding
        self.label = label
        self.confidence = 1.0
        self.mistakes = 0
        self.links = {}

    def update_confidence(self, delta):
        self.confidence = max(0.0, self.confidence + delta)

    def increment_mistakes(self):
        self.mistakes += 1

    def decay(self, rate=0.99):
        self.confidence *= rate

# ---------------------------- Neuron Network ----------------------------
class NeuronNetwork:
    def __init__(self, db_path=None):
        self.neurons = {}
        self.candidates = {}
        self.db_path = db_path
        if db_path:
            self._init_db()
    def create_candidate(self, name="UNKNOWN"):
        candidate = CandidateIdentity(name)

        self.candidates[candidate.id] = candidate

        return candidate.id


    def get_candidate(self, cid):
        return self.candidates.get(cid)


    def remove_candidate(self, cid):
        if cid in self.candidates:
            del self.candidates[cid]


    def promote_candidate(self, cid):
        candidate = self.candidates.get(cid)

        if not candidate:
            return None

        if len(candidate.embeddings) == 0:
            return None

        avg_embedding = np.mean(
            candidate.embeddings,
            axis=0
        )

        nid = self.create_neuron(
            avg_embedding,
            candidate.name
        )

        del self.candidates[cid]

        return nid
    def print_candidates(self):

        print("\n===== CANDIDATES =====")

        for cid, candidate in self.candidates.items():

            print(
                f"ID={cid} | "
                f"Name={candidate.name} | "
                f"Confidence={candidate.confidence:.2f} | "
                f"Embeddings={len(candidate.embeddings)} | "
                f"Sources={len(candidate.sources)} | "
                f"Status={candidate.status}"
            )
        
    @staticmethod
    def cosine_similarity(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

    def create_neuron(self, embedding, label):
        n = Neuron(embedding, label)
        self.neurons[n.id] = n
        # Link similar neurons
        for nid, neuron in self.neurons.items():
            if nid == n.id: 
                continue
            sim = self.cosine_similarity(n.embedding, neuron.embedding)
            if sim > 0.7:
                self.link_neurons(n.id, nid, weight=sim)
        return n.id

    def find_similar(self, embedding, threshold=0.7):
        best_nid = None
        best_sim = -1
        for nid, neuron in self.neurons.items():
            sim = self.cosine_similarity(neuron.embedding, embedding)
            if sim > threshold and sim > best_sim:
                best_sim = sim
                best_nid = nid
        return best_nid

    def update_neuron(self, nid, correct=True, new_embedding=None):
        neuron = self.neurons[nid]
        if correct:
            neuron.update_confidence(+1)
        else:
            neuron.update_confidence(-1)
            neuron.increment_mistakes()
        if new_embedding is not None:
            neuron.embedding = 0.9 * neuron.embedding + 0.1 * new_embedding

    def link_neurons(self, nid1, nid2, weight=1.0):
        self.neurons[nid1].links[nid2] = weight
        self.neurons[nid2].links[nid1] = weight

    # Decay and prune
    def decay_and_prune(self, min_confidence=0.1):
        to_delete = []
        for nid, neuron in self.neurons.items():
            neuron.decay()
            if neuron.confidence < min_confidence:
                to_delete.append(nid)
        for nid in to_delete:
            for linked_id in list(self.neurons[nid].links.keys()):
                if linked_id in self.neurons:
                    self.neurons[linked_id].links.pop(nid, None)
            self.neurons.pop(nid)

    # ---------------------------- SQLite persistence ----------------------------
    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, timeout=10)
        self.conn.execute("PRAGMA journal_mode=WAL;")  # better concurrency
        c = self.conn.cursor()
        c.execute('''
            CREATE TABLE IF NOT EXISTS neurons (
                id INTEGER PRIMARY KEY,
                label TEXT,
                confidence REAL,
                mistakes INTEGER,
                embedding TEXT,
                links TEXT
            )
        ''')
        self.conn.commit()

    import time

    def save(self, retries=5):
        if not self.db_path:
            raise ValueError("No db_path set for saving.")

        for attempt in range(retries):
            try:
                c = self.conn.cursor()
                for nid, neuron in self.neurons.items():
                    emb_json = json.dumps(neuron.embedding.tolist())
                    links_dict = {str(k): float(v) for k, v in neuron.links.items()}
                    links_json = json.dumps(links_dict)

                    c.execute('''
                        INSERT OR REPLACE INTO neurons (id, label, confidence, mistakes, embedding, links)
                        VALUES (?, ?, ?, ?, ?, ?)
                    ''', (nid, neuron.label, neuron.confidence, neuron.mistakes, emb_json, links_json))

                self.conn.commit()
                return  # success

            except sqlite3.OperationalError as e:
                if "locked" in str(e):
                    print(f"⚠️ DB locked, retrying... ({attempt+1})")
                    time.sleep(1)
                else:
                    raise e
    def close(self):
        if hasattr(self, "conn"):
            self.conn.close()

    def load(self):
        if not self.db_path:
            raise ValueError("No db_path set for loading.")
        c = self.conn.cursor()
        c.execute('SELECT id, label, confidence, mistakes, embedding, links FROM neurons')
        rows = c.fetchall()
        self.neurons = {}
        for row in rows:
            nid, label, conf, mistakes, emb_json, links_json = row
            neuron = Neuron(np.array(json.loads(emb_json)), label)
            neuron.id = nid
            neuron.confidence = conf
            neuron.mistakes = mistakes
            neuron.links = json.loads(links_json)
            self.neurons[nid] = neuron

    def print_summary(self):
        for nid, n in self.neurons.items():
            print(f"Neuron {nid}: label={n.label}, conf={n.confidence:.2f}, mistakes={n.mistakes}, links={list(n.links.keys())}")

# ---------------------------- Face Recognition + RL ----------------------------
def load_dataset(folder_path):
    X, y = [], []
    for person_name in os.listdir(folder_path):
        person_path = os.path.join(folder_path, person_name)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.endswith(('.jpg', '.png')):
                    emb = get_face_embedding(os.path.join(person_path, img_file))
                    if emb is not None:
                        X.append(emb)
                        y.append(person_name)
    return np.array(X), np.array(y)
def enrich_candidate(nn, cid, search_results):

    candidate = nn.get_candidate(cid)

    if not candidate:
        return

    for r in search_results:

        # 1. store source
        candidate.add_source(r["url"])

        # 2. store embedding (from image or model)
        candidate.add_embedding(r["image_embedding"])

        # 3. increase confidence slowly
        candidate.update_confidence(0.2)

    print(f"Candidate {cid} enriched:")
    print(f"- Sources: {len(candidate.sources)}")
    print(f"- Embeddings: {len(candidate.embeddings)}")
    print(f"- Confidence: {candidate.confidence:.2f}")
def try_promote_candidate(nn, cid, threshold=0.75):

    candidate = nn.get_candidate(cid)

    if not candidate:
        return None

    if candidate.confidence >= threshold and len(candidate.embeddings) >= 3:

        avg_embedding = np.mean(candidate.embeddings, axis=0)

        nid = nn.create_neuron(avg_embedding, candidate.name)

        candidate.promote()

        nn.remove_candidate(cid)

        print(f"🔥 Candidate promoted to neuron: {nid}")

        return nid

    return None
    
def train_svm(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    clf = SVC(kernel='rbf', probability=True)
    clf.fit(X_scaled, y_train)
    return clf, scaler
    
def decide_action(result):

    confidence = result["svm_confidence"]
    similarity = result["similarity"]

    if (
        confidence >= 0.90
        and similarity >= 0.85
    ):
        return "TRUST"

    elif (
        confidence >= 0.60
        or similarity >= 0.70
    ):
        return "VERIFY"

    return "SEARCH"
        
def predict_face(nn: NeuronNetwork, svm_clf, scaler, img_path, true_label=None):
    emb = get_face_embedding(img_path)

    if emb is None:
        return {
            "prediction": "NO_FACE",
            "svm_confidence": 0.0,
            "neuron_id": None,
            "similarity": 0.0,
        }

    # ----------------------------
    # SVM Prediction
    # ----------------------------
    scaled_emb = scaler.transform([emb])

    pred_label = svm_clf.predict(scaled_emb)[0]

    probs = svm_clf.predict_proba(scaled_emb)[0]

    max_prob = float(np.max(probs))

    if max_prob < 0.60:
        pred_label = "UNKNOWN"

    # ----------------------------
    # Find Similar Neuron
    # ----------------------------
    nid = nn.find_similar(emb)

    best_similarity = 0.0

    if nid is not None:
        best_similarity = float(
            nn.cosine_similarity(
                nn.neurons[nid].embedding,
                emb
            )
        )

    # ----------------------------
    # RL / Brain Update
    # ----------------------------
    if pred_label == "UNKNOWN":

        if nid is not None:

            nn.update_neuron(
                nid,
                correct=False,
                new_embedding=emb
            )

        else:

            nid = nn.create_neuron(
                emb,
                "UNKNOWN"
            )

        return {
            "prediction": "UNKNOWN",
            "svm_confidence": max_prob,
            "neuron_id": nid,
            "similarity": best_similarity,
        }

    # Known prediction
    if nid is not None:

        correct = (
            pred_label == true_label
            if true_label
            else True
        )

        nn.update_neuron(
            nid,
            correct=correct,
            new_embedding=emb
        )

    return {
        "prediction": pred_label,
        "svm_confidence": max_prob,
        "neuron_id": nid,
        "similarity": best_similarity,
    }
 
# ---------------------------- Example Usage ----------------------------
if __name__ == "__main__":

    # ----------------------------
    # Init search engine
    # ----------------------------
    search_engine = SearchEngine()

    train_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\train"
    test_folder = r"C:\Users\mhd_gamer\Documents\GitHub\face_recognition\FACE DATASET\Faces\test"

    # Load dataset
    X_train, y_train = load_dataset(train_folder)
    X_test, y_test = load_dataset(test_folder)
    print(f"✅ Loaded {len(X_train)} train and {len(X_test)} test images")

    # Train SVM
    svm_clf, scaler = train_svm(X_train, y_train)
    y_pred = svm_clf.predict(scaler.transform(X_test))
    print(f"✅ SVM Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

    # Save model
    joblib.dump(svm_clf, "svm_face_recognition.pkl")
    joblib.dump(scaler, "svm_scaler.pkl")
    print("✅ SVM model and scaler saved!")

    # Initialize brain
    nn = NeuronNetwork(db_path="face_neurons.sqlite")

    for emb, label in zip(X_train, y_train):
        nn.create_neuron(emb, label)

    # Test prediction
    example_img = os.path.join(test_folder, "img1.jpg")

    result = predict_face(
        nn,
        svm_clf,
        scaler,
        example_img,
        true_label="Akshay Kumar"
    )

    action = decide_action(result)

    if action == "SEARCH":

        print("\n===== SEARCH TRIGGERED =====")

        cid = nn.create_candidate()
        print(f"Created candidate: {cid}")

        results = search_engine.search("unknown person")

        enrich_candidate(nn, cid, results)

        try_promote_candidate(nn, cid)

    # Output
    print("\n===== BRAIN DECISION =====")
    print(f"Prediction     : {result['prediction']}")
    print(f"SVM Confidence : {result['svm_confidence']:.4f}")
    print(f"Similarity     : {result['similarity']:.4f}")
    print(f"Neuron ID      : {result['neuron_id']}")
    print(f"Action         : {action}")

    nn.print_candidates()
    nn.save()
    nn.close()

C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\utils\detect_face.py:250: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if method is "Min":
C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\utils\detect_face.py:250: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if method is "Min":
C:\Users\mhd_gamer\.conda\envs\gemma\lib\site-packages\facenet_pytorch\models\utils\detect_face.py:250: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if method is "Min":


KeyboardInterrupt: 

In [18]:
    example_img = os.path.join(test_folder, "images.jpg")
    result = predict_face(
    nn,
    svm_clf,
    scaler,
    example_img,
    true_label="Akshay Kumar"
)

    action = decide_action(result)
    if action == "SEARCH":

        cid = nn.create_candidate()

    

    
        print(
            f"Created Candidate: {cid}"
        )
    print("\n===== BRAIN DECISION =====")
    print(f"Prediction     : {result['prediction']}")
    print(f"SVM Confidence : {result['svm_confidence']:.4f}")
    print(f"Similarity     : {result['similarity']:.4f}")
    print(f"Neuron ID      : {result['neuron_id']}")
    print(f"Action         : {action}")
    # Save neuron brain
    nn.print_candidates()

   


===== BRAIN DECISION =====
Prediction     : UNKNOWN
SVM Confidence : 0.0618
Similarity     : 1.0000
Neuron ID      : 673691075
Action         : VERIFY

===== CANDIDATES =====
